# Intro to Python for Data Analysis
## Alabama NO2 Air Quality Dataset (2024)

**Dataset:** 488 daily NO2 observations from two EPA monitoring sites in Jefferson County, Alabama  
**Time range:** January 1 – September 9, 2024  
**Key column:** `Daily Max 1-hour NO2 Concentration` (ppb)  
**EPA standard:** The NAAQS 1-hour NO2 limit is **53 ppb**

---

### Workshop outline
1. Load & inspect the data
2. Line chart — NO2 over time by site
3. Box plot — distribution by month and site
4. Bar chart — monthly average NO2 per site
5. Scatter plot — NO2 vs. AQI with correlation

---
## Step 1: Load & inspect the data

We use **pandas** to load the CSV and parse the date column into a proper datetime format.  
We also rename the long column names to shorter aliases we'll use throughout.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import numpy as np

# Load the CSV
# encoding='utf-8-sig' handles the byte-order mark (BOM) that Excel sometimes adds
df = pd.read_csv('alabamaData_NO2.csv', encoding='utf-8-sig')

# Parse the Date column from M/D/YY text format (e.g. '1/15/24') into a real datetime
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%y')

# Rename the long column names we'll use most often
df = df.rename(columns={
    'Daily Max 1-hour NO2 Concentration': 'no2',
    'Daily AQI Value':                    'aqi',
    'Local Site Name':                    'site'
})

# Add helper columns for time-based grouping
df['month']       = df['Date'].dt.month
df['month_name']  = df['Date'].dt.strftime('%b')
df['day_of_week'] = df['Date'].dt.day_name()

print('Shape:', df.shape)
print('Sites:', df['site'].unique())
print('Date range:', df['Date'].min().date(), 'to', df['Date'].max().date())

In [ ]:
# Quick statistical summary of our two main columns
df[['no2', 'aqi']].describe().round(2)

In [ ]:
# Preview the first few rows
df[['Date', 'site', 'no2', 'aqi', 'month_name']].head(10)

### Key observations from Step 1
- **488 rows**, 2 monitoring sites: *North Birmingham* and *Arkadelphia/Near Road*
- NO2 ranges from **2.4 to 55.1 ppb**, with a mean of ~21.6 ppb
- AQI ranges from **2 to 52** — all readings fall in the Good or Moderate category
- The Pearson r between NO2 and AQI should be very close to 1.0 (we'll verify in Step 5)

---
## Step 2: Line chart — NO2 over time by site

We plot each monitoring site as a separate line using `groupby()` + a `for` loop.  
A dashed red reference line marks the EPA NAAQS 1-hour standard of 53 ppb.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))

colors = {
    'North Birmingham':      '#1f77b4',
    'Arkadelphia/Near Road': '#ff7f0e'
}

for site_name, group in df.groupby('site'):
    group_sorted = group.sort_values('Date')
    ax.plot(
        group_sorted['Date'],
        group_sorted['no2'],
        label=site_name,
        color=colors[site_name],
        linewidth=1.2,
        alpha=0.85
    )

# EPA NAAQS 1-hour NO2 standard reference line
ax.axhline(
    y=53,
    color='red',
    linestyle='--',
    linewidth=1,
    label='NAAQS 1-hr standard (53 ppb)'
)

# Format the x-axis to show month labels cleanly
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))

ax.set_title('Daily Max NO2 Concentration — Jefferson County, AL (2024)', fontsize=13)
ax.set_ylabel('NO2 (ppb)')
ax.set_xlabel('')
ax.legend(loc='upper right', fontsize=10)
ax.grid(axis='y', linewidth=0.5, alpha=0.4)

plt.tight_layout()
plt.savefig('plot1_timeseries.png', dpi=150)
plt.show()

### Discussion — Line chart
- NO2 is **highest in January–February** and lower in summer months
- This is consistent with **winter temperature inversions** — a meteorological effect where cold air near the ground traps pollutants instead of letting them disperse upward
- One spike at North Birmingham nearly touches the 53 ppb NAAQS standard
- **Arkadelphia** only has data for Jan–Feb and reappears briefly in September — a good reminder to always check for gaps in your data

---
## Step 3: Box plot — distribution by month and site

A box plot shows the **full spread** of values, not just the average.  
The box spans the IQR (25th–75th percentile), the center line is the median, and dots are outliers.

In [ ]:
# Order months chronologically Jan through Sep
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep']

fig, ax = plt.subplots(figsize=(13, 5))

sns.boxplot(
    data=df,
    x='month_name',
    y='no2',
    hue='site',
    order=month_order,
    palette='Set2',
    linewidth=0.8,
    fliersize=3,
    ax=ax
)

ax.axhline(
    y=53,
    color='red',
    linestyle='--',
    linewidth=1,
    label='NAAQS 1-hr standard (53 ppb)'
)

ax.set_title('NO2 Distribution by Month and Site — 2024', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Daily Max NO2 (ppb)')
ax.legend(title='Site', fontsize=10)
ax.grid(axis='y', linewidth=0.5, alpha=0.4)

# Annotate what the box parts mean
ax.text(
    0.01, 0.97,
    'Box = IQR (25th–75th percentile)  |  Line = median  |  Dots = outliers',
    transform=ax.transAxes, fontsize=9, color='gray', va='top'
)

plt.tight_layout()
plt.savefig('plot2_boxplot.png', dpi=150)
plt.show()

### Discussion — Box plot
- North Birmingham shows a **wider spread** and more outlier dots in winter months, indicating more variability in cold-weather pollution events
- Summer months (Jun–Aug) are tighter and lower — more stable, cleaner air
- Arkadelphia only has boxes for Jan, Feb, and Sep — the missing months appear as empty space, making the data gaps visually obvious
- Box plots are better than bar charts when you want to communicate **spread and outliers**, not just averages

---
## Step 4: Bar chart — monthly average NO2 per site

We first aggregate the data using `groupby().mean()` — the pandas equivalent of `GROUP BY` + `AVG()` in SQL.  
Then we plot the grouped averages as a side-by-side bar chart.

In [ ]:
# Aggregate: average NO2 per month per site
# SQL equivalent: SELECT month, site, AVG(no2) FROM no2_data GROUP BY month, site
monthly_avg = (
    df.groupby(['month', 'month_name', 'site'])['no2']
    .mean()
    .reset_index()
    .sort_values('month')
)

# Print the table so students can see the numbers behind the chart
pivot = monthly_avg.pivot(index='month_name', columns='site', values='no2').round(2)
pivot = pivot.reindex(month_order)
print('Monthly average NO2 (ppb) by site:')
print(pivot.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

sns.barplot(
    data=monthly_avg,
    x='month_name',
    y='no2',
    hue='site',
    order=month_order,
    palette='Set2',
    edgecolor='white',
    linewidth=0.5,
    ax=ax
)

# Label each bar with its rounded value
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f', fontsize=8, padding=2)

ax.set_title('Average Monthly NO2 by Site — 2024', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Avg Daily Max NO2 (ppb)')
ax.legend(title='Site', fontsize=10)
ax.grid(axis='y', linewidth=0.5, alpha=0.4)
ax.set_ylim(0, ax.get_ylim()[1] * 1.12)  # extra headroom for bar labels

plt.tight_layout()
plt.savefig('plot3_bar.png', dpi=150)
plt.show()

### Discussion — Bar chart
- **February** is the peak month for both sites (~24 ppb)
- `groupby().mean().reset_index()` is the pandas equivalent of `GROUP BY` + `AVG()` in SQL — `reset_index()` converts the grouped result back into a flat DataFrame
- The missing Arkadelphia bars (Apr–Aug) confirm the data gap we saw in the time series
- Bar charts are best when you want to **compare a single summary statistic** (the mean) across categories

---
## Step 5: Scatter plot — NO2 vs. AQI with correlation

We plot each day as a dot (NO2 on x-axis, AQI on y-axis), colored by site.  
We add a linear trend line for each site using `np.polyfit()`, and compute the Pearson correlation.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

sns.scatterplot(
    data=df,
    x='no2',
    y='aqi',
    hue='site',
    palette='Set2',
    alpha=0.6,
    s=35,
    edgecolor='none',
    ax=ax
)

# Add a linear regression trend line for each site
for site_name, group in df.groupby('site'):
    m, b = np.polyfit(group['no2'], group['aqi'], 1)
    x_range = np.array([group['no2'].min(), group['no2'].max()])
    ax.plot(x_range, m * x_range + b, linewidth=1.5, linestyle='--', alpha=0.8)

# Pearson correlation coefficient
corr = df[['no2', 'aqi']].corr().iloc[0, 1]
ax.text(
    0.05, 0.95,
    f'Pearson r = {corr:.3f}',
    transform=ax.transAxes, fontsize=10, va='top',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='lightgray')
)

ax.set_title('NO2 Concentration vs. AQI Value', fontsize=13)
ax.set_xlabel('Daily Max NO2 (ppb)')
ax.set_ylabel('Daily AQI Value')
ax.legend(title='Site', fontsize=10)
ax.grid(linewidth=0.5, alpha=0.3)

plt.tight_layout()
plt.savefig('plot4_scatter.png', dpi=150)
plt.show()

In [ ]:
# Full correlation matrix across NO2, AQI, and month
print('Pearson r (no2 vs aqi):', round(corr, 4))
print()
print('Full correlation matrix:')
df[['no2', 'aqi', 'month']].corr().round(3)

### Discussion — Scatter plot
- Pearson r = **0.999** — an almost perfect positive correlation
- This makes sense: the EPA AQI for NO2 is **calculated directly from the NO2 concentration**, so they are essentially the same measurement on different scales
- The two sites' dots overlap almost completely, meaning both monitors follow the same mathematical relationship between NO2 and AQI
- Month has essentially **zero correlation** with either NO2 or AQI (r ≈ 0.016), confirming that the calendar month alone is not a strong predictor — it's the weather patterns within those months that drive pollution levels

---
## Bonus: Summary statistics table

Reproduce the Excel summary table from Part 1 entirely in pandas.

In [ ]:
summary = pd.DataFrame({
    'Metric': [
        'Average NO2 (ppb)',
        'Max NO2 (ppb)',
        'Min NO2 (ppb)',
        'Std Dev NO2 (ppb)',
        'Count of days',
        'Average AQI',
        'Days with AQI > 50',
        'Days with AQI > 100',
        '% of days with AQI > 50'
    ],
    'Value': [
        round(df['no2'].mean(), 2),
        round(df['no2'].max(), 1),
        round(df['no2'].min(), 1),
        round(df['no2'].std(), 2),
        len(df),
        round(df['aqi'].mean(), 2),
        (df['aqi'] > 50).sum(),
        (df['aqi'] > 100).sum(),
        f"{(df['aqi'] > 50).mean() * 100:.1f}%"
    ]
})

summary

In [ ]:
# Compare sites side by side — same as the Excel pivot table
site_summary = df.groupby('site')['no2'].agg(
    avg_no2='mean',
    max_no2='max',
    min_no2='min',
    std_no2='std',
    days='count'
).round(2)

site_summary

---
## Key takeaways

| Concept | pandas syntax | SQL equivalent |
|---|---|---|
| Load data | `pd.read_csv()` | `SELECT * FROM table` |
| Filter rows | `df[df['col'] > value]` | `WHERE col > value` |
| Group & aggregate | `df.groupby('col')['val'].mean()` | `GROUP BY col` + `AVG(val)` |
| Sort | `.sort_values('col')` | `ORDER BY col` |
| Count rows | `len(df)` or `.count()` | `COUNT(*)` |
| New column | `df['new'] = ...` | computed column in `SELECT` |

### Libraries used
- **pandas** — data loading, cleaning, and group-level aggregation
- **matplotlib** — low-level chart building and axis formatting
- **seaborn** — high-level statistical visualizations (box plots, bar charts, scatter)
- **numpy** — numerical operations 
